# Session 43: Model Interpretability — Permutation Importance & SHAP
**Week 4 Permutation Importance vs. Built-In Random Forest Feature Importance**

[cite_start]In this notebook, we calculate test-set **permutation importance** for a Random Forest Regressor ($n\_estimators=300$, $random\_state=42$) predicting student final grades (`G3`) without using `G1` and `G2` (early-warning setup) [cite: 6865, 6873-6874]. [cite_start]We compare permutation importance against built-in Gini importance to evaluate model dependency on unseen test data [cite: 7323-7327].

In [4]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Set random seed
RANDOM_STATE = 42

# Resolve project directories
PROJECT_ROOT = Path.cwd().resolve()
for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (parent / ".git").exists():
        PROJECT_ROOT = parent
        break

DATA_DIRECTORY = PROJECT_ROOT / "data"
INTERPRETABILITY_DIR = PROJECT_ROOT / "reports" / "interpretability"
FIGURES_DIR = PROJECT_ROOT / "figures"

INTERPRETABILITY_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project Root:", PROJECT_ROOT)
print("Libraries imported successfully.")

Project Root: /home/nikhil/Desktop/VSCode/GSSRP/student-performance-prediction-ml
Libraries imported successfully.


In [5]:
def load_and_preprocess_data():
    # Prefer the prepared early-warning dataset (G1/G2 already excluded),
    # falling back to the raw semicolon-delimited UCI file.
    candidates = [
        (DATA_DIRECTORY / "processed" / "early_warning_dataset.csv", ","),
        (DATA_DIRECTORY / "processed" / "student-mat-clean.csv", ";"),
        (DATA_DIRECTORY / "raw" / "student-mat.csv", ";"),
    ]

    TARGET = "G3"

    for path, sep in candidates:
        if not path.exists():
            continue

        df = pd.read_csv(path, sep=sep)

        # Reject target-only files (e.g. y_train_full.csv), which otherwise
        # pass a bare "G3 in columns" check and yield a zero-column X.
        if TARGET not in df.columns or df.shape[1] < 2:
            continue

        EXCLUDED = [c for c in ["G1", "G2"] if c in df.columns]

        X = df.drop(columns=[TARGET] + EXCLUDED).copy()
        y = df[TARGET].copy()

        numeric_features = X.select_dtypes(include=np.number).columns.tolist()
        categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.20, random_state=RANDOM_STATE
        )

        preprocessor = ColumnTransformer(
            transformers=[
                ("numeric", "passthrough", numeric_features),
                ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
            ],
            remainder="drop",
            verbose_feature_names_out=False
        )

        Xtr_array = preprocessor.fit_transform(X_train)
        Xte_array = preprocessor.transform(X_test)
        feature_names = preprocessor.get_feature_names_out()

        Xtr_f = pd.DataFrame(Xtr_array, columns=feature_names, index=X_train.index)
        Xte_f = pd.DataFrame(Xte_array, columns=feature_names, index=X_test.index)

        print(f"Loaded source: {path.relative_to(PROJECT_ROOT)}")
        return Xtr_f, Xte_f, y_train, y_test

    raise FileNotFoundError("Could not locate a student dataset containing G3 plus predictors.")

Xtr_f, Xte_f, y_train, y_test = load_and_preprocess_data()
assert Xtr_f.shape[1] > 0, "Preprocessing produced zero feature columns."
print(f"Transformed Training shape: {Xtr_f.shape}, Test shape: {Xte_f.shape}")

Loaded source: data/processed/early_warning_dataset.csv
Transformed Training shape: (316, 39), Test shape: (79, 39)


In [6]:
# Train Random Forest regressor with 300 estimators
rf = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(Xtr_f, y_train)

# Evaluate model baseline test performance
y_pred = rf.predict(Xte_f)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Test MAE:  {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")
print(f"Test R²:   {r2:.4f}")

Test MAE:  3.0727
Test RMSE: 3.8589
Test R²:   0.2738


In [7]:
# Compute permutation importance on test set across 10 repetitions
r = permutation_importance(
    estimator=rf,
    X=Xte_f,
    y=y_test,
    scoring="neg_root_mean_squared_error",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

permutation_results = pd.DataFrame({
    "feature": Xte_f.columns,
    "permutation_importance_mean": r.importances_mean,
    "permutation_importance_std": r.importances_std
}).sort_values(by="permutation_importance_mean", ascending=False).reset_index(drop=True)

permutation_results.insert(0, "permutation_rank", range(1, len(permutation_results) + 1))

print("Top 10 Predictors by Permutation Importance:")
display(permutation_results.head(10).round(4))

Top 10 Predictors by Permutation Importance:


,permutation_rank,feature,permutation_importance_mean,permutation_importance_std
0,1,failures,0.6690,0.1793
1,2,absences,0.4058,0.1403
2,3,goout,0.1658,0.0628
3,4,sex_M,0.1429,0.0524
4,5,schoolsup_yes,0.1007,0.0358
5,6,Medu,0.0866,0.0428
6,7,age,0.0603,0.0267
7,8,guardian_other,0.0470,0.0380
8,9,Fjob_teacher,0.0370,0.0149
9,10,Mjob_health,0.0344,0.0194


In [8]:
# Extract built-in tree importances
built_in_results = pd.DataFrame({
    "feature": Xtr_f.columns,
    "built_in_importance": rf.feature_importances_
}).sort_values(by="built_in_importance", ascending=False).reset_index(drop=True)

built_in_results.insert(0, "built_in_rank", range(1, len(built_in_results) + 1))

# Merge tables
comparison = built_in_results.merge(permutation_results, on="feature", how="inner")
comparison["rank_difference"] = (comparison["built_in_rank"] - comparison["permutation_rank"]).abs()

# Normalize importances
built_in_max = comparison["built_in_importance"].max()
perm_max = comparison["permutation_importance_mean"].clip(lower=0).max()

comparison["built_in_importance_normalized"] = comparison["built_in_importance"] / built_in_max
comparison["permutation_importance_normalized"] = comparison["permutation_importance_mean"].clip(lower=0) / (perm_max if perm_max > 0 else 1.0)

def classify_rank_comparison(diff):
    if diff <= 2:
        return "Strong agreement"
    elif diff <= 5:
        return "Moderate disagreement"
    else:
        return "Strong disagreement"

comparison["comparison_status"] = comparison["rank_difference"].apply(classify_rank_comparison)

print("Top Comparison Rows:")
display(comparison.head(10).round(4))

Top Comparison Rows:


,built_in_rank,feature,built_in_importance,permutation_rank,permutation_importance_mean,permutation_importance_std,rank_difference,built_in_importance_normalized,permutation_importance_normalized,comparison_status
0,1,absences,0.1900,2,0.4058,0.1403,1,1.0000,0.6066,Strong agreement
1,2,failures,0.1460,1,0.6690,0.1793,1,0.7684,1.0000,Strong agreement
2,3,health,0.0520,39,-0.0478,0.0464,36,0.2737,0.0000,Strong disagreement
3,4,goout,0.0497,3,0.1658,0.0628,1,0.2613,0.2479,Strong agreement
4,5,age,0.0412,7,0.0603,0.0267,2,0.2168,0.0902,Strong agreement
5,6,studytime,0.0332,14,0.0209,0.0298,8,0.1748,0.0312,Strong disagreement
6,7,traveltime,0.0330,12,0.0229,0.0361,5,0.1734,0.0342,Moderate disagreement
7,8,freetime,0.0330,21,0.0083,0.0227,13,0.1734,0.0124,Strong disagreement
8,9,Fedu,0.0284,34,-0.0124,0.0214,25,0.1497,0.0000,Strong disagreement
9,10,Walc,0.0281,16,0.0149,0.0140,6,0.1479,0.0223,Strong disagreement


In [9]:
# 1. Permutation Importance Plot
top10_perm = permutation_results.head(10).sort_values(by="permutation_importance_mean", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(top10_perm["feature"], top10_perm["permutation_importance_mean"], xerr=top10_perm["permutation_importance_std"], color="darkorange", edgecolor="black", alpha=0.85, capsize=4)
plt.axvline(0, color="black", linewidth=1)
plt.xlabel("Decrease in Test Performance After Shuffling (RMSE)")
plt.ylabel("Predictor")
plt.title("Top 10 Predictors by Permutation Importance")
plt.grid(axis="x", linestyle="--", alpha=0.35)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "permutation_importance.png", dpi=300, bbox_inches="tight")
plt.close()

# 2. Built-in vs. Permutation Side-by-Side Plot
shared_features = set(built_in_results.head(10)["feature"]).union(set(permutation_results.head(10)["feature"]))
plot_data = comparison[comparison["feature"].isin(shared_features)].sort_values(by="permutation_importance_normalized", ascending=True)

y_positions = np.arange(len(plot_data))
bar_height = 0.38

plt.figure(figsize=(12, max(7, len(plot_data) * 0.55)))
plt.barh(y_positions - bar_height / 2, plot_data["built_in_importance_normalized"], height=bar_height, label="Built-in Importance", color="steelblue", edgecolor="black")
plt.barh(y_positions + bar_height / 2, plot_data["permutation_importance_normalized"], height=bar_height, label="Permutation Importance", color="darkorange", edgecolor="black")
plt.yticks(y_positions, plot_data["feature"])
plt.xlabel("Normalized Importance")
plt.ylabel("Predictor")
plt.title("Built-In Versus Permutation Importance")
plt.legend()
plt.grid(axis="x", linestyle="--", alpha=0.35)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "built_in_vs_permutation_importance.png", dpi=300, bbox_inches="tight")
plt.close()

print("Saved figures under figures/")

Saved figures under figures/


In [10]:
# Save CSV artifacts
permutation_results.to_csv(INTERPRETABILITY_DIR / "permutation_importance.csv", index=False)
permutation_results.head(10).to_csv(INTERPRETABILITY_DIR / "top_10_permutation_importance.csv", index=False)
comparison.to_csv(INTERPRETABILITY_DIR / "interpretability_comparison.csv", index=False)

# Build and write text summary note
top_builtin_feat = built_in_results.iloc[0]["feature"]
top_perm_feat = permutation_results.iloc[0]["feature"]
shared_top10 = sorted(set(built_in_results.head(10)["feature"]).intersection(set(permutation_results.head(10)["feature"])))

rank_corr = comparison[["built_in_rank", "permutation_rank"]].corr(method="spearman").iloc[0, 1]

note_text = f"""SESSION 43: INTERPRETABILITY COMPARISON NOTE
============================================
Target: G3 (Early-Warning Setup, G1 & G2 Excluded)
Model: Random Forest Regressor (n_estimators=300, random_state=42)

MAIN RESULTS:
Top Built-in Predictor: {top_builtin_feat}
Top Permutation Predictor: {top_perm_feat}
Shared Top-10 Predictors ({len(shared_top10)}): {', '.join(shared_top10)}
Spearman Rank Correlation: {rank_corr:.4f}

INTERPRETATION:
- Built-in Gini importance describes how frequently and effectively the Random Forest used a predictor when constructing tree splits during training.
- Permutation importance directly measures how much test-set RMSE deteriorates when a feature is randomly shuffled on unseen test data.
- Disagreements arise when features are correlated, redundant, or when continuous/encoded features offer additional splitting opportunities during training.
"""

with open(INTERPRETABILITY_DIR / "interpretability_comparison_note.txt", "w", encoding="utf-8") as f:
    f.write(note_text)

print("Exported CSVs and interpretability_comparison_note.txt under reports/interpretability/")

Exported CSVs and interpretability_comparison_note.txt under reports/interpretability/


In [11]:
# Verify all required files exist and contain non-zero size
req_files = [
    INTERPRETABILITY_DIR / "permutation_importance.csv",
    INTERPRETABILITY_DIR / "top_10_permutation_importance.csv",
    INTERPRETABILITY_DIR / "interpretability_comparison.csv",
    INTERPRETABILITY_DIR / "interpretability_comparison_note.txt",
    FIGURES_DIR / "permutation_importance.png",
    FIGURES_DIR / "built_in_vs_permutation_importance.png"
]

for filepath in req_files:
    assert filepath.exists(), f"Missing artifact: {filepath}"
    assert filepath.stat().st_size > 0, f"Empty artifact: {filepath}"

print("=" * 72)
print("SESSION 43 INTERPRETABILITY DELIVERABLE COMPLETED SUCCESSFULLY")
print("=" * 72)

SESSION 43 INTERPRETABILITY DELIVERABLE COMPLETED SUCCESSFULLY
